# 09 — Tableau Data Preparation

## Objective

The goal of this notebook is to prepare clean,
analysis-ready exports for the final Tableau dashboard.

The dashboard will follow the project storytelling:

**Understand → Segment → Predict → Decision Support**

The Tableau layer will combine:
- Steam market overview,
- market segmentation results,
- segment performance,
- and supervised-learning insights.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

processed_path = project_root / "data/processed"

tableau_path = project_root / "data/tableau"
tableau_path.mkdir(parents=True, exist_ok=True)

In [2]:
# Load cleaned historical dataset
df_games = pd.read_csv(
    processed_path / "games_cleaned_march2025.csv"
)

# Load market-segmentation results
df_segments = pd.read_csv(
    processed_path / "steam_market_segments_2025.csv"
)

print("Historical games:", df_games.shape)
print("Segmentation data:", df_segments.shape)

Historical games: (94948, 48)
Segmentation data: (94948, 18)


In [3]:
# Add clustering results to the cleaned historical dataset

segment_columns = [
    "appid",
    "cluster_4",
    "segment_name"
]

df_tableau = df_games.merge(
    df_segments[segment_columns],
    on="appid",
    how="left",
    validate="one_to_one"
)

print("Merged dataset:", df_tableau.shape)
print(
    "Missing segment labels:",
    df_tableau["segment_name"].isna().sum()
)

Merged dataset: (94948, 50)
Missing segment labels: 0


In [4]:
# Create Tableau-friendly business fields

df_tableau["reach_status"] = np.select(
    [
        df_tableau["owners_lower"].isna(),
        df_tableau["owners_lower"] >= 100_000
    ],
    [
        "Unknown",
        "High Reach (100k+)"
    ],
    default="Below 100k"
)

df_tableau["review_coverage"] = np.where(
    df_tableau["num_reviews_total"] >= 50,
    "50+ Reviews",
    "Limited Review Data"
)

df_tableau["satisfaction_status"] = np.select(
    [
        df_tableau["num_reviews_total"] < 50,
        df_tableau["pct_pos_total"] >= 80
    ],
    [
        "Insufficient Reviews",
        "High Satisfaction (80%+)"
    ],
    default="Below 80%"
)

df_tableau["free_or_paid"] = np.where(
    df_tableau["is_free"],
    "Free",
    "Paid"
)

In [5]:
tableau_columns = [
    "appid",
    "name",
    "release_year",
    "price",
    "price_band",
    "free_or_paid",
    "genres",

    "cluster_4",
    "segment_name",

    "owners_lower",
    "owners_upper",
    "estimated_owners_midpoint",
    "reach_status",

    "pct_pos_total",
    "num_reviews_total",
    "review_coverage",
    "satisfaction_status",

    "median_playtime_hours",
    "peak_ccu",
    "achievements",
    "dlc_count",

    "windows",
    "mac",
    "linux"
]

tableau_games = df_tableau[tableau_columns].copy()

display(tableau_games.head())

print("Tableau dataset shape:", tableau_games.shape)
print("Unique games:", tableau_games["appid"].nunique())
print("Segments:", tableau_games["segment_name"].nunique())

,appid,name,release_year,price,price_band,free_or_paid,genres,cluster_4,segment_name,owners_lower,...,num_reviews_total,review_coverage,satisfaction_status,median_playtime_hours,peak_ccu,achievements,dlc_count,windows,mac,linux
0,730,Counter-Strike 2,2012,0.00,Free,Free,"[""Action"", ""Free To Play""]",2,Established Multi-Platform Titles,100000000.0,...,8632939.0,50+ Reviews,High Satisfaction (80%+),86.233333,1212356,1,1,True,False,True
1,578080,PUBG: BATTLEGROUNDS,2017,0.00,Free,Free,"[""Action"", ""Adventure"", ""Massively Multiplayer...",1,Core Paid-Oriented Titles,50000000.0,...,2513842.0,50+ Reviews,Below 80%,0.000000,616738,37,0,True,False,False
2,570,Dota 2,2013,0.00,Free,Free,"[""Action"", ""Strategy"", ""Free To Play""]",2,Established Multi-Platform Titles,200000000.0,...,2452595.0,50+ Reviews,High Satisfaction (80%+),14.966667,555977,0,2,True,True,True
3,271590,Grand Theft Auto V Legacy,2015,0.00,Free,Free,"[""Action"", ""Adventure""]",1,Core Paid-Oriented Titles,50000000.0,...,1803832.0,50+ Reviews,High Satisfaction (80%+),118.350000,117698,77,0,True,False,False
4,488824,Tom Clancy's Rainbow Six® Siege,2015,19.99,$10-$19.99,Paid,"[""Action""]",3,DLC-Oriented Content Titles,0.0,...,1168404.0,50+ Reviews,High Satisfaction (80%+),0.000000,0,0,9,True,False,False


Tableau dataset shape: (94948, 24)
Unique games: 94948
Segments: 4


In [6]:
games_output = tableau_path / "tableau_games.csv"

tableau_games.to_csv(
    games_output,
    index=False
)

print("Saved:", games_output)

Saved: /Users/AurianeB/inside-steam-analytics/data/tableau/tableau_games.csv


In [7]:
import ast

# Convert genre strings back to Python lists
def parse_genres(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []


genre_long = df_tableau[
    [
        "appid",
        "name",
        "release_year",
        "price_band",
        "genres",
        "segment_name",
        "reach_status",
        "free_or_paid"
    ]
].copy()

genre_long["genre"] = (
    genre_long["genres"]
    .apply(parse_genres)
)

genre_long = (
    genre_long
    .explode("genre")
    .drop(columns="genres")
)

genre_long["genre"] = genre_long["genre"].fillna("Unknown")

print("Genre records:", len(genre_long))
print("Unique games:", genre_long["appid"].nunique())
print("Unique genres:", genre_long["genre"].nunique())

display(genre_long.head(10))

Genre records: 263698
Unique games: 94948
Unique genres: 34


,appid,name,release_year,price_band,segment_name,reach_status,free_or_paid,genre
0,730,Counter-Strike 2,2012,Free,Established Multi-Platform Titles,High Reach (100k+),Free,Action
0,730,Counter-Strike 2,2012,Free,Established Multi-Platform Titles,High Reach (100k+),Free,Free To Play
1,578080,PUBG: BATTLEGROUNDS,2017,Free,Core Paid-Oriented Titles,High Reach (100k+),Free,Action
1,578080,PUBG: BATTLEGROUNDS,2017,Free,Core Paid-Oriented Titles,High Reach (100k+),Free,Adventure
1,578080,PUBG: BATTLEGROUNDS,2017,Free,Core Paid-Oriented Titles,High Reach (100k+),Free,Massively Multiplayer
1,578080,PUBG: BATTLEGROUNDS,2017,Free,Core Paid-Oriented Titles,High Reach (100k+),Free,Free To Play
2,570,Dota 2,2013,Free,Established Multi-Platform Titles,High Reach (100k+),Free,Action
2,570,Dota 2,2013,Free,Established Multi-Platform Titles,High Reach (100k+),Free,Strategy
2,570,Dota 2,2013,Free,Established Multi-Platform Titles,High Reach (100k+),Free,Free To Play
3,271590,Grand Theft Auto V Legacy,2015,Free,Core Paid-Oriented Titles,High Reach (100k+),Free,Action


In [8]:
genre_output = tableau_path / "tableau_game_genres.csv"

genre_long.to_csv(
    genre_output,
    index=False
)

print("Saved:", genre_output)

Saved: /Users/AurianeB/inside-steam-analytics/data/tableau/tableau_game_genres.csv


In [9]:
model_metrics = pd.DataFrame({
    "model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "precision_high_reach": [
        0.158,
        0.201
    ],
    "recall_high_reach": [
        0.641,
        0.586
    ],
    "f1_high_reach": [
        0.254,
        0.299
    ],
    "roc_auc": [
        0.687,
        0.710
    ],
    "average_precision": [
        0.189,
        0.239
    ]
})

display(model_metrics)

,model,precision_high_reach,recall_high_reach,f1_high_reach,roc_auc,average_precision
0,Logistic Regression,0.158,0.641,0.254,0.687,0.189
1,Random Forest,0.201,0.586,0.299,0.710,0.239


In [10]:
metrics_output = (
    tableau_path
    / "tableau_model_metrics.csv"
)

model_metrics.to_csv(
    metrics_output,
    index=False
)

print("Saved:", metrics_output)

Saved: /Users/AurianeB/inside-steam-analytics/data/tableau/tableau_model_metrics.csv


In [11]:
expected_files = [
    "tableau_games.csv",
    "tableau_game_genres.csv",
    "tableau_model_metrics.csv",
    "tableau_feature_importance.csv"
]

print("TABLEAU EXPORT CHECK\n")

for file_name in expected_files:
    file_path = tableau_path / file_name

    if file_path.exists():
        print(f"OK: {file_name}")
    else:
        print(f"MISSING: {file_name}")

TABLEAU EXPORT CHECK

OK: tableau_games.csv
OK: tableau_game_genres.csv
OK: tableau_model_metrics.csv
OK: tableau_feature_importance.csv
